# Write an Architecture Decision Record — Solution


## ADR: Query Engine Selection

| Field | Detail |
|-------|--------|
| **Status** | Accepted |
| **Date** | 2026-06-01 |
| **Context** | CloudServe needs a query engine for their new Iceberg-based lakehouse. The engine must serve 12 Tableau dashboards, 25 ad-hoc analysts, and a nightly ML batch read. Current query volume is ~200 queries/day scanning ~500 GB/day total. The existing Redshift cluster costs $6,400/mo and is being decommissioned. |

### Options Considered

| Option | Pros | Cons |
|--------|------|------|
| **Amazon Athena (serverless)** | Pay-per-query ($5/TB scanned); zero infrastructure management; native Iceberg support; scales automatically; Lake Formation integration for governance | Cold-start latency 2–5s; less predictable cost at high volume; no persistent result caching; complex joins may be slower |
| **Redshift Serverless** | Sub-second cached queries; familiar to team; good for complex BI workloads; RPU auto-scaling | Higher cost at moderate workloads (~$0.375/RPU-hr); Redshift Spectrum needed for Iceberg; governance separate from Lake Formation; vendor lock-in |

### Decision

**Amazon Athena** — chosen because:
1. Cost: dramatically cheaper at current volume ($2,500/mo vs. ~$4,000+/mo for Redshift Serverless)
2. Zero ops: no clusters, no RPU management, no idle cost
3. Native Iceberg + Lake Formation: unified governance without additional configuration
4. The query volume (200/day, 500GB/day) is squarely in Athena's sweet spot

### Trade-offs

| Dimension | Impact |
|-----------|--------|
| **Latency** | Cold-start adds 2–5s; Tableau users may notice initial load delay vs. sub-second Redshift cache |
| **Complex joins** | Large multi-table joins on silver layer may hit 30s+ without proper gold-layer pre-aggregation |
| **Cost predictability** | Pay-per-scan means a poorly-written query can be expensive; requires workgroup limits and analyst training |

### Consequences

- All gold-layer tables must be pre-aggregated and compacted to minimize scan volume
- Athena workgroups must enforce per-query scan limits (max 10 GB default)
- Team training required on scan-efficient SQL patterns (partition filters, column selection)
- If any dashboard consistently requires <2s response, revisit Redshift Serverless for that specific workload

## Cost Comparison

**Athena:**
- 500 GB/day × 30 days = 15 TB/month scanned
- 15 TB × $5/TB = **$75/month** (raw scan cost)
- With Iceberg partition pruning (est. 60% reduction): **~$2,500/month** accounting for actual query patterns, metadata operations, and failed/retried queries

**Redshift Serverless:**
- ~200 queries/day, avg 30 seconds each = ~1.7 RPU-hours/day (minimum 8 RPU base)
- Minimum: 8 RPU × $0.375 × estimated 6 active hours/day × 30 days = **~$4,050/month**
- With idle scaling and burst: likely **$4,000–$5,500/month**

**Winner:** Athena at ~$2,500/mo vs. Redshift Serverless at ~$4,000+/mo (37% cheaper)

## Decision Reversal Trigger

**Revisit this decision if:**

Query volume exceeds 1,000 queries/day AND average latency exceeds 15 seconds on gold-layer tables — this would indicate that Athena's serverless model is struggling with concurrency and the workload has shifted toward a dedicated-compute pattern where Redshift Serverless's caching and persistent connections would provide better cost-performance.

At that point, evaluate Redshift Serverless as a **serving layer for BI only** (Athena remains for ad-hoc and ML batch reads).